# Week 9 — Trustworthy ML: Leakage, Calibration, Coverage and Domain Shift

**Contact:** 3 hours lecture  
**Course:** MSc Astrostatistics for the Survey Era

## Learning notebooks from the Crete school
- `../../10_ML_Practices/ML_Practices.ipynb`

These Crete notebooks are **learning material**. This lecture notebook supplies the semester narrative, deeper astronomical interpretation, and the binary-SMBH case study.

## Three-hour pacing
- **0:00–0:20** — recap, motivating scientific question, diagnostic poll
- **0:20–1:05** — theory block I + worked derivation
- **1:05–1:15** — short break / discussion
- **1:15–2:00** — theory block II + computational demonstration
- **2:00–2:10** — short break
- **2:10–2:45** — binary-SMBH case study / failure mode
- **2:45–3:00** — synthesis, exit questions, bridge to the 4-hour practical

## Learning objectives

- Audit direct, target, temporal and group leakage.
- Evaluate probabilistic calibration rather than only point accuracy.
- Distinguish aleatoric and epistemic uncertainty conceptually.
- Design in-distribution and out-of-distribution tests.
- Explain the simulation-to-real gap and what evidence is required before scientific deployment.

## 1. Scientific ML can fail while benchmarks look excellent


A model is trustworthy only relative to a clearly stated deployment distribution and decision. Random-split test performance can be excellent because train and test share the same simulator assumptions, cadence distribution and selection effects.

In survey astronomy, deployment may involve a new field, higher redshift, different S/N, a changed pipeline or real sources that violate the simulator family. These are forms of **domain shift**.

The correct question is therefore not only "how accurate is the model?" but "under what changes does the claim remain valid?"

> **Discussion checkpoint.** Identify the estimand, observed data, and strongest modelling assumption in the example just discussed.

### Worked computational demonstration — IID versus redshift-shift split

Run the cell, then change one assumption and explain what changes statistically.

In [ ]:
import numpy as np, sys
sys.path.append('../src')
from course_utils import load_scalar_sample
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error

df=load_scalar_sample()
features=['inputs__mass_ratio','inputs__eccentricity','inputs__Eddington_fraction']
X=df[features]
y=np.log10(df['inputs__period'])
train=df['inputs__redshift']<2
test=df['inputs__redshift']>=2
m=RandomForestRegressor(n_estimators=150,random_state=2026,n_jobs=-1).fit(X[train],y[train])
pred=m.predict(X[test])
print('shifted RMSE:',mean_squared_error(y[test],pred)**.5)

## 2. Leakage taxonomy


**Target leakage** occurs when a feature is constructed using information unavailable at inference time or directly derived from the target. **Group leakage** occurs when related observations appear across splits. **Preprocessing leakage** occurs when scaling, imputation or feature selection is fitted using test data. **Temporal leakage** occurs when future information is used to predict the past.

The binary-SMBH Parquet is especially dangerous because physical truths and observable arrays live in the same record. A one-line column selection mistake can turn an inverse problem into an answer lookup.

### Worked computational demonstration — Empirical interval coverage helper

Run the cell, then change one assumption and explain what changes statistically.

In [ ]:
def coverage(y,lo,hi):
    y=np.asarray(y); lo=np.asarray(lo); hi=np.asarray(hi)
    return np.mean((y>=lo)&(y<=hi))
# Example use after a model supplies intervals:
# print(coverage(y_true, lo68, hi68))

## 3. Calibration and coverage


For probabilistic classification, a calibrated model satisfies approximately

$$
P(Y=1\mid \hat p=p)\approx p.
$$

For interval prediction, a nominal $100(1-\alpha)\%$ interval should contain the true value in about that fraction of repeated comparable cases:

$$
P(\theta_{\rm true}\in C_{1-\alpha}(x))\approx1-\alpha.
$$

Coverage should also be checked conditionally: a model can have correct global 68% coverage while being overconfident at high redshift and underconfident at low redshift.

### Worked computational demonstration — Reliability diagram skeleton

Run the cell, then change one assumption and explain what changes statistically.

In [ ]:
def reliability_bins(y, p, nbins=10):
    y=np.asarray(y); p=np.asarray(p)
    edges=np.linspace(0,1,nbins+1)
    out=[]
    for lo,hi in zip(edges[:-1],edges[1:]):
        mask=(p>=lo)&(p<hi if hi<1 else p<=hi)
        if mask.sum():
            out.append((p[mask].mean(),y[mask].mean(),mask.sum()))
    return out
# Students: plot predicted probability against empirical frequency.

## 4. Aleatoric versus epistemic uncertainty


Aleatoric uncertainty describes irreducible variation in the observation model; epistemic uncertainty describes uncertainty due to limited knowledge or data/model support. The separation is model-dependent, but the conceptual distinction is useful.

A network that outputs a Gaussian variance may learn heteroscedastic aleatoric noise, yet it can still be dangerously confident on out-of-distribution inputs. Ensembles, Bayesian approximations or density/OOD diagnostics can help reveal epistemic uncertainty, but none substitutes for stress testing.

## 5. Domain-shift experiments


A strong MSc exercise deliberately creates a shift. Train on $z<2$ and test on $z>2$; train on high-cadence fields and test on lower cadence; perturb photometric uncertainties; remove a mechanism from training and introduce it at test time.

Report both in-distribution and shifted performance. Then inspect where uncertainty widens—or fails to widen. A model that degrades gracefully and signals uncertainty is more scientifically valuable than one that wins the IID benchmark by a small margin.

## 6. Simulation-to-real validation ladder


Before applying an inference model trained on simulations to real LSST AGN, one would want a ladder of evidence: simulator checks against known physical behaviour, recovery on held-out simulations, stress tests across plausible simulator variations, comparison with simpler methods, validation on real benchmark objects where independent constraints exist, and posterior predictive checks in observable space.

The final project requires students to articulate this ladder rather than claiming that success on synthetic data alone validates a real-survey measurement.

## Deep dive — types of distribution shift

It helps to distinguish several shifts:

- **covariate shift:** $p(x)$ changes while $p(y\mid x)$ is stable;
- **label shift:** $p(y)$ changes while $p(x\mid y)$ is stable;
- **concept shift:** the conditional relation $p(y\mid x)$ changes;
- **support shift:** deployment contains regions absent from training.

Simulation-to-real differences can include all four.

### Recalibration

If a probabilistic classifier is discriminative but miscalibrated, post-hoc methods such as Platt/logistic scaling or isotonic regression can improve calibration on a representative calibration set. But recalibration learned on IID simulations may fail under domain shift.

### Conditional coverage

Global interval coverage

$$
P(Y\in C(X))=1-\alpha
$$

can hide subgroup failures. Plot coverage as a function of redshift, S/N, cadence or predicted uncertainty. A scientifically dangerous model is often one whose worst calibration occurs exactly where the most interesting objects live.

### Conformal prediction

Conformal methods can provide finite-sample marginal coverage under exchangeability with relatively weak model assumptions. They are worth introducing conceptually because they separate predictive model quality from an interval-calibration layer. However, their guarantees also depend on the calibration and deployment data being exchangeable; domain shift again breaks the simple guarantee.

## Binary-SMBH synthesis questions

Answer verbally before leaving the lecture.

1. Which quantity in today's topic is a **property of the simulator**, and which could be an **observable property of a real survey**?
2. What uncertainty or selection effect would most strongly distort the result?
3. What diagnostic would you require before trusting the inference?
4. What information would be unavailable if these were real LSST sources rather than simulations?

## Key equations / ideas to retain

Create your own 6–10 line summary here during class.

## Further reading

- Crete `10_ML_Practices/ML_Practices.ipynb`.
- General ML calibration and dataset-shift literature.
- Course Week 6 for selection-induced domain shift.

# Part II — Extended lecture development

## 14. Expected calibration error and Brier score

For binary probability predictions, the Brier score is

$$
{\rm BS}=\frac1N\sum_i(\hat p_i-y_i)^2.
$$

It rewards both discrimination and calibration.

Expected calibration error (ECE) bins predictions and averages the absolute
difference between mean predicted probability and empirical frequency. ECE is
easy to communicate but depends on binning.

Reliability diagrams should therefore accompany any scalar calibration score.

## 15. Proper scoring rules

A proper scoring rule is optimised in expectation by reporting the true
probabilistic belief. Examples include log loss and Brier score.

This matters because accuracy encourages only the final class decision, while a
proper score rewards honest probabilities.

For scientific downstream decisions, a well-calibrated probability can be more
useful than a slightly more accurate hard classifier.

## 16. Regression calibration beyond central intervals

For a full predictive CDF $F_i(y)$, probability integral transform values

$$
u_i=F_i(y_i)
$$

should be approximately uniform under a calibrated continuous predictive model.

PIT histograms can reveal:
- U-shape: predictions too narrow;
- central hump: predictions too broad;
- slope/asymmetry: bias.

This is closely related to SBC rank diagnostics in Week 10.

## 17. Ensembles and epistemic uncertainty

Train several models under different initialisations/bootstrap samples. Their
spread can act as a practical measure of model instability.

If all ensemble members agree on an unsupported OOD example, however, the
ensemble can still be confidently wrong. Ensemble disagreement is a useful
diagnostic, not a proof of epistemic coverage.

The strongest OOD analysis combines support diagnostics, deliberate shifts and
predictive uncertainty.

## 18. Data augmentation versus domain realism

Augmentation can improve robustness if it represents plausible measurement
variation. Adding Gaussian noise, perturbing cadence or masking observations can
teach invariances.

But unrealistic augmentation can harm inference or create a false sense of
robustness. Augmentation parameters should be motivated by the observed/supplied
data rather than arbitrary image-ML conventions.

## 19. Domain adaptation does not remove the need for validation

Techniques such as importance weighting, feature alignment or fine-tuning can
adapt models to a shifted domain.

Yet adaptation assumes something about what changed. If the conditional
physical mapping changes because the simulator is missing a mechanism, simple
covariate reweighting is insufficient.

State which shift assumption your adaptation addresses.

## 20. Stress-testing a scientific model

Useful stress tests include:
- higher redshift;
- larger photometric noise;
- fewer epochs;
- different field;
- target extremes;
- missing bands;
- perturbed feature distributions.

The point is not to produce one "robustness score." It is to map where the
scientific inference remains valid and where it breaks.

In [ ]:
# Simple feature-shift diagnostic using standardized distance
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import NearestNeighbors

df=load_scalar_sample()
features=["inputs__mass_ratio","inputs__eccentricity",
          "inputs__Eddington_fraction"]
X=df[features].astype(float).to_numpy()
z=df["inputs__redshift"].astype(float).to_numpy()

train=z<2
test=z>=2

sc=StandardScaler().fit(X[train])
Xt=sc.transform(X[train])
Xs=sc.transform(X[test])

nn=NearestNeighbors(n_neighbors=1).fit(Xt)
dist,_=nn.kneighbors(Xs)

print("median nearest-training distance in shifted set:",np.median(dist))
print("95th percentile:",np.quantile(dist,.95))

## 21. Selective prediction / abstention

A trustworthy model may decline to make a strong prediction when uncertainty
or OOD score is high.

Instead of forcing a prediction for every object, define a retained subset
satisfying a confidence/support criterion and evaluate:
- retained fraction;
- error/calibration on retained sources;
- bias introduced by abstention.

Abstention is itself a selection process and should be documented.

## 22. Model cards for scientific ML

A compact model card for an astronomy model should state:
- training simulation/data source;
- target and feature definitions;
- known support/range;
- split strategy;
- IID metrics;
- calibration;
- tested shifts;
- known failure cases;
- prohibited uses.

This is a useful final-project deliverable because it turns statistical caveats
into explicit operational documentation.

# Part III — Worked trustworthy-ML problems

## Problem A — perfect global coverage, bad science

Imagine two equal subgroups:
- group A has 95% coverage for a nominal 68% interval;
- group B has 41% coverage.

The global coverage is 68%.

A single aggregate number says "perfectly calibrated," while one subgroup is
dramatically overconfident.

Therefore always inspect conditional coverage along scientifically meaningful
axes.

## Problem B — distinguish shift types

For each scenario, classify the shift:

1. Same physical relation, but high-redshift sources are more common.
2. Same class prevalence, but a new cadence changes feature distributions.
3. A physical mechanism absent from simulations changes the light-curve/target
   relationship.
4. Real data contain feature values outside all simulations.

Discuss why the correct mitigation differs across label, covariate, concept and
support shift.

## Problem C — abstention changes the sample

A model refuses predictions when OOD score is high. Accuracy improves on the
remaining sample.

Is the method now "better"?

Only conditionally. Report retained fraction and which objects are removed.
Abstention may preferentially discard high-redshift or rare systems, creating a
scientifically biased usable sample.

# Part IV — Additional trustworthy-ML seminar

## Dataset shift can invalidate threshold choices

Suppose a classifier threshold was selected to achieve 90% purity in validation.
If class prevalence changes at deployment, the same threshold can have very
different precision even if the conditional score distributions remain stable.

This is a concrete consequence of label/prevalence shift. Operational
thresholds must therefore be monitored, not frozen forever.

## Calibration is conditional on the evaluation population

A probability model can be calibrated on WFD-like objects and miscalibrated on
DDF-like objects. A single reliability diagram averaged over both may hide the
problem.

For the final project, ask students to show calibration/coverage against at
least one context variable. The objective is not to obtain perfectly flat
diagnostics everywhere but to **identify where the uncertainty statement ceases
to be reliable**.

## Reproducibility of ML evaluation

Record the exact random split seed, grouping rule, preprocessing pipeline and
hyperparameter search space. Otherwise a reported improvement may be an
accidental property of one favourable split.